In [1]:
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
import os
import pandas as pd

load_dotenv(find_dotenv(), override=True)

print("DB_NAME =", os.getenv("DB_NAME"))

DB_NAME = Comp640_db


In [2]:
from sqlalchemy import create_engine
import os

engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

In [3]:
import pandas as pd

pd.read_sql("SELECT * FROM clinic_location;", engine)

,location_id,name,address_line1,address_line2,city,state,zip_code,phone,email,timezone,is_active,created_at
0,a0000001-0000-0000-0000-000000000001,Meridian Health – Downtown LA,350 S Grand Ave,None,Los Angeles,CA,90071,(213) 555-0100,downtown@meridianhealth.example,America/Los_Angeles,True,2026-04-29 02:37:01.556217+00:00
1,a0000001-0000-0000-0000-000000000002,Meridian Health – Santa Monica,1450 Ocean Ave,None,Santa Monica,CA,90401,(310) 555-0200,santamonica@meridianhealth.example,America/Los_Angeles,True,2026-04-29 02:37:01.556217+00:00


# 1.Find the next available appointment slots for a doctor

In [4]:
#Clininc Location 
pd.read_sql("SELECT * FROM doctor;", engine)

,doctor_id,first_name,last_name,specialty,license_number,email,phone,location_id,is_active,created_at
0,c0000001-0000-0000-0000-000000000001,Eleanor,Voss,Internal Medicine,CA-MD-100001,evoss@meridianhealth.example,(213) 555-0101,a0000001-0000-0000-0000-000000000001,True,2026-04-29 02:37:01.556217+00:00
1,c0000001-0000-0000-0000-000000000002,Marcus,Tan,Cardiology,CA-MD-100002,mtan@meridianhealth.example,(213) 555-0102,a0000001-0000-0000-0000-000000000001,True,2026-04-29 02:37:01.556217+00:00
2,c0000001-0000-0000-0000-000000000003,Priya,Nair,Family Medicine,CA-MD-100003,pnair@meridianhealth.example,(310) 555-0201,a0000001-0000-0000-0000-000000000002,True,2026-04-29 02:37:01.556217+00:00
3,c0000001-0000-0000-0000-000000000004,David,Okafor,Endocrinology,CA-MD-100004,dokafor@meridianhealth.example,(310) 555-0202,a0000001-0000-0000-0000-000000000002,True,2026-04-29 02:37:01.556217+00:00


In [5]:
#Finding the avaiability time for Dr.Tan
# Available on Tuesdays and Thursdays in Downtown LA from 1pm-5pm 
query="""
SELECT *
FROM doctor_availability
WHERE doctor_id='c0000001-0000-0000-0000-000000000002'
"""
pd.read_sql(query, engine)

,availability_id,doctor_id,location_id,day_of_week,slot_start,slot_end,slot_date,is_booked,is_blocked,created_at
0,d0000001-0000-0000-0000-000000000004,c0000001-0000-0000-0000-000000000002,a0000001-0000-0000-0000-000000000001,2,13:00:00,17:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00
1,d0000001-0000-0000-0000-000000000005,c0000001-0000-0000-0000-000000000002,a0000001-0000-0000-0000-000000000001,4,13:00:00,17:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00


# 2.Clinic utilization report

In [6]:
appointment_df=pd.read_sql("SELECT * FROM appointment;", engine)
appointment_df

,appointment_id,patient_id,doctor_id,location_id,room_id,scheduled_at,duration_mins,appointment_type,status,telehealth_url,chief_complaint,notes,created_at
0,aa000001-0000-0000-0000-000000000001,e0000001-0000-0000-0000-000000000001,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,b0000001-0000-0000-0000-000000000001,2025-05-12 16:00:00+00:00,60,in_person,completed,None,Persistent fatigue and shortness of breath,"BP 138/88, ordered CBC and CMP",2026-04-29 02:37:01.556217+00:00
1,aa000001-0000-0000-0000-000000000002,e0000001-0000-0000-0000-000000000002,c0000001-0000-0000-0000-000000000003,a0000001-0000-0000-0000-000000000002,None,2025-05-14 17:00:00+00:00,30,telehealth,completed,https://meet.meridian.example/rm/XJ9KL2,Medication review — metformin titration,Increased metformin to 1000mg BID,2026-04-29 02:37:01.556217+00:00
2,aa000001-0000-0000-0000-000000000003,e0000001-0000-0000-0000-000000000003,c0000001-0000-0000-0000-000000000002,a0000001-0000-0000-0000-000000000001,b0000001-0000-0000-0000-000000000002,2025-05-20 21:00:00+00:00,45,in_person,completed,None,Palpitations and exertional chest tightness,ECG ordered; lipid panel requested,2026-04-29 02:37:01.556217+00:00
3,aa000001-0000-0000-0000-000000000004,e0000001-0000-0000-0000-000000000004,c0000001-0000-0000-0000-000000000004,a0000001-0000-0000-0000-000000000002,None,2025-05-21 18:00:00+00:00,30,telehealth,completed,https://meet.meridian.example/rm/AO5TR8,Type 2 diabetes quarterly check-in,HbA1c trending down; continue current regimen,2026-04-29 02:37:01.556217+00:00
4,aa000001-0000-0000-0000-000000000005,e0000001-0000-0000-0000-000000000005,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,b0000001-0000-0000-0000-000000000001,2025-05-28 16:30:00+00:00,30,in_person,completed,None,Annual wellness exam,All vitals normal; advised flu vaccine,2026-04-29 02:37:01.556217+00:00
5,aa000001-0000-0000-0000-000000000006,e0000001-0000-0000-0000-000000000006,c0000001-0000-0000-0000-000000000003,a0000001-0000-0000-0000-000000000002,b0000001-0000-0000-0000-000000000005,2025-06-02 15:30:00+00:00,60,in_person,completed,None,New patient — hypertension and osteoporosis ma...,Referred for DEXA scan; BP meds reviewed,2026-04-29 02:37:01.556217+00:00
6,aa000001-0000-0000-0000-000000000008,e0000001-0000-0000-0000-000000000008,c0000001-0000-0000-0000-000000000002,a0000001-0000-0000-0000-000000000001,b0000001-0000-0000-0000-000000000002,2026-06-12 21:30:00+00:00,45,in_person,confirmed,None,Post-cardiac stress test review,None,2026-04-29 02:37:01.556217+00:00
7,aa000001-0000-0000-0000-000000000009,e0000001-0000-0000-0000-000000000009,c0000001-0000-0000-0000-000000000004,a0000001-0000-0000-0000-000000000002,None,2025-06-05 17:00:00+00:00,30,telehealth,cancelled,None,Thyroid follow-up,Patient cancelled — rescheduling,2026-04-29 02:37:01.556217+00:00
8,aa000001-0000-0000-0000-000000000010,e0000001-0000-0000-0000-000000000010,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,b0000001-0000-0000-0000-000000000001,2025-06-04 17:00:00+00:00,30,in_person,completed,None,Follow-up — iron-deficiency anaemia,Haemoglobin improving; continue iron supplemen...,2026-04-29 02:37:01.556217+00:00
9,aa000001-0000-0000-0000-000000000007,e0000001-0000-0000-0000-000000000007,c0000001-0000-0000-0000-000000000003,a0000001-0000-0000-0000-000000000002,None,2026-06-10 16:00:00+00:00,30,telehealth,confirmed,https://meet.meridian.example/rm/DM3QW6,Anxiety follow-up,None,2026-04-29 02:37:01.556217+00:00


In [19]:
availability_df=pd.read_sql("SELECT * FROM doctor_availability;", engine)
availability_df

,availability_id,doctor_id,location_id,day_of_week,slot_start,slot_end,slot_date,is_booked,is_blocked,created_at
0,d0000001-0000-0000-0000-000000000001,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,1,09:00:00,12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00
1,d0000001-0000-0000-0000-000000000002,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,3,09:00:00,12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00
2,d0000001-0000-0000-0000-000000000003,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,5,09:00:00,12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00
3,d0000001-0000-0000-0000-000000000004,c0000001-0000-0000-0000-000000000002,a0000001-0000-0000-0000-000000000001,2,13:00:00,17:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00
4,d0000001-0000-0000-0000-000000000005,c0000001-0000-0000-0000-000000000002,a0000001-0000-0000-0000-000000000001,4,13:00:00,17:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00
5,d0000001-0000-0000-0000-000000000006,c0000001-0000-0000-0000-000000000003,a0000001-0000-0000-0000-000000000002,1,08:00:00,17:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00
6,d0000001-0000-0000-0000-000000000007,c0000001-0000-0000-0000-000000000003,a0000001-0000-0000-0000-000000000002,2,08:00:00,17:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00
7,d0000001-0000-0000-0000-000000000008,c0000001-0000-0000-0000-000000000003,a0000001-0000-0000-0000-000000000002,3,08:00:00,17:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00
8,d0000001-0000-0000-0000-000000000009,c0000001-0000-0000-0000-000000000003,a0000001-0000-0000-0000-000000000002,4,08:00:00,17:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00
9,d0000001-0000-0000-0000-000000000010,c0000001-0000-0000-0000-000000000003,a0000001-0000-0000-0000-000000000002,5,08:00:00,17:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00


In [20]:
merged = availability_df.merge(
    appointment_df,
    on=["doctor_id", "location_id"],
    how="left"
)
merged

,availability_id,doctor_id,location_id,day_of_week,slot_start,slot_end,slot_date,is_booked,is_blocked,created_at_x,...,patient_id,room_id,scheduled_at,duration_mins,appointment_type,status,telehealth_url,chief_complaint,notes,created_at_y
0,d0000001-0000-0000-0000-000000000001,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,1,09:00:00,12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00,...,e0000001-0000-0000-0000-000000000001,b0000001-0000-0000-0000-000000000001,2025-05-12 16:00:00+00:00,60,in_person,completed,None,Persistent fatigue and shortness of breath,"BP 138/88, ordered CBC and CMP",2026-04-29 02:37:01.556217+00:00
1,d0000001-0000-0000-0000-000000000001,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,1,09:00:00,12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00,...,e0000001-0000-0000-0000-000000000005,b0000001-0000-0000-0000-000000000001,2025-05-28 16:30:00+00:00,30,in_person,completed,None,Annual wellness exam,All vitals normal; advised flu vaccine,2026-04-29 02:37:01.556217+00:00
2,d0000001-0000-0000-0000-000000000001,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,1,09:00:00,12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00,...,e0000001-0000-0000-0000-000000000010,b0000001-0000-0000-0000-000000000001,2025-06-04 17:00:00+00:00,30,in_person,completed,None,Follow-up — iron-deficiency anaemia,Haemoglobin improving; continue iron supplemen...,2026-04-29 02:37:01.556217+00:00
3,d0000001-0000-0000-0000-000000000002,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,3,09:00:00,12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00,...,e0000001-0000-0000-0000-000000000001,b0000001-0000-0000-0000-000000000001,2025-05-12 16:00:00+00:00,60,in_person,completed,None,Persistent fatigue and shortness of breath,"BP 138/88, ordered CBC and CMP",2026-04-29 02:37:01.556217+00:00
4,d0000001-0000-0000-0000-000000000002,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,3,09:00:00,12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00,...,e0000001-0000-0000-0000-000000000005,b0000001-0000-0000-0000-000000000001,2025-05-28 16:30:00+00:00,30,in_person,completed,None,Annual wellness exam,All vitals normal; advised flu vaccine,2026-04-29 02:37:01.556217+00:00
5,d0000001-0000-0000-0000-000000000002,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,3,09:00:00,12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00,...,e0000001-0000-0000-0000-000000000010,b0000001-0000-0000-0000-000000000001,2025-06-04 17:00:00+00:00,30,in_person,completed,None,Follow-up — iron-deficiency anaemia,Haemoglobin improving; continue iron supplemen...,2026-04-29 02:37:01.556217+00:00
6,d0000001-0000-0000-0000-000000000003,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,5,09:00:00,12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00,...,e0000001-0000-0000-0000-000000000001,b0000001-0000-0000-0000-000000000001,2025-05-12 16:00:00+00:00,60,in_person,completed,None,Persistent fatigue and shortness of breath,"BP 138/88, ordered CBC and CMP",2026-04-29 02:37:01.556217+00:00
7,d0000001-0000-0000-0000-000000000003,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,5,09:00:00,12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00,...,e0000001-0000-0000-0000-000000000005,b0000001-0000-0000-0000-000000000001,2025-05-28 16:30:00+00:00,30,in_person,completed,None,Annual wellness exam,All vitals normal; advised flu vaccine,2026-04-29 02:37:01.556217+00:00
8,d0000001-0000-0000-0000-000000000003,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,5,09:00:00,12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00,...,e0000001-0000-0000-0000-000000000010,b0000001-0000-0000-0000-000000000001,2025-06-04 17:00:00+00:00,30,in_person,completed,None,Follow-up — iron-deficiency anaemia,Haemoglobin improving; continue iron supplemen

In [21]:
merged["slot_start"] = pd.to_datetime("2025-05-01 " + merged["slot_start"].astype(str))
merged["slot_end"]   = pd.to_datetime("2025-05-01 " + merged["slot_end"].astype(str))
merged["duration"] = merged["slot_end"] - merged["slot_start"]


In [22]:
merged["slot_start"]

0    2025-05-01 09:00:00
1    2025-05-01 09:00:00
2    2025-05-01 09:00:00
3    2025-05-01 09:00:00
4    2025-05-01 09:00:00
5    2025-05-01 09:00:00
6    2025-05-01 09:00:00
7    2025-05-01 09:00:00
8    2025-05-01 09:00:00
9    2025-05-01 13:00:00
10   2025-05-01 13:00:00
11   2025-05-01 13:00:00
12   2025-05-01 13:00:00
13   2025-05-01 08:00:00
14   2025-05-01 08:00:00
15   2025-05-01 08:00:00
16   2025-05-01 08:00:00
17   2025-05-01 08:00:00
18   2025-05-01 08:00:00
19   2025-05-01 08:00:00
20   2025-05-01 08:00:00
21   2025-05-01 08:00:00
22   2025-05-01 08:00:00
23   2025-05-01 08:00:00
24   2025-05-01 08:00:00
25   2025-05-01 08:00:00
26   2025-05-01 08:00:00
27   2025-05-01 08:00:00
28   2025-05-01 08:00:00
29   2025-05-01 08:00:00
Name: slot_start, dtype: datetime64[ns]

In [23]:
merged["slot_end"]

0    2025-05-01 12:00:00
1    2025-05-01 12:00:00
2    2025-05-01 12:00:00
3    2025-05-01 12:00:00
4    2025-05-01 12:00:00
5    2025-05-01 12:00:00
6    2025-05-01 12:00:00
7    2025-05-01 12:00:00
8    2025-05-01 12:00:00
9    2025-05-01 17:00:00
10   2025-05-01 17:00:00
11   2025-05-01 17:00:00
12   2025-05-01 17:00:00
13   2025-05-01 17:00:00
14   2025-05-01 17:00:00
15   2025-05-01 17:00:00
16   2025-05-01 17:00:00
17   2025-05-01 17:00:00
18   2025-05-01 17:00:00
19   2025-05-01 17:00:00
20   2025-05-01 17:00:00
21   2025-05-01 17:00:00
22   2025-05-01 17:00:00
23   2025-05-01 17:00:00
24   2025-05-01 17:00:00
25   2025-05-01 17:00:00
26   2025-05-01 17:00:00
27   2025-05-01 17:00:00
28   2025-05-01 17:00:00
29   2025-05-01 17:00:00
Name: slot_end, dtype: datetime64[ns]

In [24]:
merged["duration"]

0    0 days 03:00:00
1    0 days 03:00:00
2    0 days 03:00:00
3    0 days 03:00:00
4    0 days 03:00:00
5    0 days 03:00:00
6    0 days 03:00:00
7    0 days 03:00:00
8    0 days 03:00:00
9    0 days 04:00:00
10   0 days 04:00:00
11   0 days 04:00:00
12   0 days 04:00:00
13   0 days 09:00:00
14   0 days 09:00:00
15   0 days 09:00:00
16   0 days 09:00:00
17   0 days 09:00:00
18   0 days 09:00:00
19   0 days 09:00:00
20   0 days 09:00:00
21   0 days 09:00:00
22   0 days 09:00:00
23   0 days 09:00:00
24   0 days 09:00:00
25   0 days 09:00:00
26   0 days 09:00:00
27   0 days 09:00:00
28   0 days 09:00:00
29   0 days 09:00:00
Name: duration, dtype: timedelta64[ns]

In [25]:
#converting hours to minutes 3 hours * 60 minutes/hour= 180 minutes 
merged["available_time"] = merged["duration"].dt.total_seconds() / 60
merged["available_time"]

0     180.0
1     180.0
2     180.0
3     180.0
4     180.0
5     180.0
6     180.0
7     180.0
8     180.0
9     240.0
10    240.0
11    240.0
12    240.0
13    540.0
14    540.0
15    540.0
16    540.0
17    540.0
18    540.0
19    540.0
20    540.0
21    540.0
22    540.0
23    540.0
24    540.0
25    540.0
26    540.0
27    540.0
28    540.0
29    540.0
Name: available_time, dtype: float64

In [26]:
# only in person
in_person_appointment=merged.loc[merged["appointment_type"] =="in_person"]
in_person_appointment

,availability_id,doctor_id,location_id,day_of_week,slot_start,slot_end,slot_date,is_booked,is_blocked,created_at_x,...,scheduled_at,duration_mins,appointment_type,status,telehealth_url,chief_complaint,notes,created_at_y,duration,available_time
0,d0000001-0000-0000-0000-000000000001,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,1,2025-05-01 09:00:00,2025-05-01 12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00,...,2025-05-12 16:00:00+00:00,60,in_person,completed,None,Persistent fatigue and shortness of breath,"BP 138/88, ordered CBC and CMP",2026-04-29 02:37:01.556217+00:00,0 days 03:00:00,180.0
1,d0000001-0000-0000-0000-000000000001,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,1,2025-05-01 09:00:00,2025-05-01 12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00,...,2025-05-28 16:30:00+00:00,30,in_person,completed,None,Annual wellness exam,All vitals normal; advised flu vaccine,2026-04-29 02:37:01.556217+00:00,0 days 03:00:00,180.0
2,d0000001-0000-0000-0000-000000000001,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,1,2025-05-01 09:00:00,2025-05-01 12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00,...,2025-06-04 17:00:00+00:00,30,in_person,completed,None,Follow-up — iron-deficiency anaemia,Haemoglobin improving; continue iron supplemen...,2026-04-29 02:37:01.556217+00:00,0 days 03:00:00,180.0
3,d0000001-0000-0000-0000-000000000002,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,3,2025-05-01 09:00:00,2025-05-01 12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00,...,2025-05-12 16:00:00+00:00,60,in_person,completed,None,Persistent fatigue and shortness of breath,"BP 138/88, ordered CBC and CMP",2026-04-29 02:37:01.556217+00:00,0 days 03:00:00,180.0
4,d0000001-0000-0000-0000-000000000002,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,3,2025-05-01 09:00:00,2025-05-01 12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00,...,2025-05-28 16:30:00+00:00,30,in_person,completed,None,Annual wellness exam,All vitals normal; advised flu vaccine,2026-04-29 02:37:01.556217+00:00,0 days 03:00:00,180.0
5,d0000001-0000-0000-0000-000000000002,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,3,2025-05-01 09:00:00,2025-05-01 12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00,...,2025-06-04 17:00:00+00:00,30,in_person,completed,None,Follow-up — iron-deficiency anaemia,Haemoglobin improving; continue iron supplemen...,2026-04-29 02:37:01.556217+00:00,0 days 03:00:00,180.0
6,d0000001-0000-0000-0000-000000000003,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,5,2025-05-01 09:00:00,2025-05-01 12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00,...,2025-05-12 16:00:00+00:00,60,in_person,completed,None,Persistent fatigue and shortness of breath,"BP 138/88, ordered CBC and CMP",2026-04-29 02:37:01.556217+00:00,0 days 03:00:00,180.0
7,d0000001-0000-0000-0000-000000000003,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,5,2025-05-01 09:00:00,2025-05-01 12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00,...,2025-05-28 16:30:00+00:00,30,in_person,completed,None,Annual wellness exam,All vitals normal; advised flu vaccine,2026-04-29 02:37:01.556217+00:00,0 days 03:00:00,180.0
8,d0000001-0000-0000-0000-000000000003,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,5,2025-05-01 09:00:00,2025-05-01 12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00,...,2025-06-04 17:00:00+00:00,30,in_person,completed,None,Follow-up — iron-deficiency anaemia,Haemoglobin improving; continue iron supplemen...,2026-04-29 02:37:01.556217+00:00,0 days 03:00:00,180.0
9,d0000001-0000-0000-0000-000000000004,c0000001-0000-0000-0000-000000000002,a0000001-0000-0000-0000-000000000001,2,2025-05-01 13:00:00,2025-05-01 17:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00,...,2025-05-20 21:00:00

In [27]:
#in person
booked_summary = in_person_appointment.groupby(
    ["location_id", "day_of_week"]
)["duration_mins"].sum().reset_index()
booked_summary

,location_id,day_of_week,duration_mins
0,a0000001-0000-0000-0000-000000000001,1,120
1,a0000001-0000-0000-0000-000000000001,2,90
2,a0000001-0000-0000-0000-000000000001,3,120
3,a0000001-0000-0000-0000-000000000001,4,90
4,a0000001-0000-0000-0000-000000000001,5,120
5,a0000001-0000-0000-0000-000000000002,1,60
6,a0000001-0000-0000-0000-000000000002,2,60
7,a0000001-0000-0000-0000-000000000002,3,60
8,a0000001-0000-0000-0000-000000000002,4,60
9,a0000001-0000-0000-0000-000000000002,5,60


In [28]:
availability_summary = merged.groupby(
    ["location_id", "day_of_week"]
)["available_time"].sum().reset_index()
availability_summary

,location_id,day_of_week,available_time
0,a0000001-0000-0000-0000-000000000001,1,540.0
1,a0000001-0000-0000-0000-000000000001,2,480.0
2,a0000001-0000-0000-0000-000000000001,3,540.0
3,a0000001-0000-0000-0000-000000000001,4,480.0
4,a0000001-0000-0000-0000-000000000001,5,540.0
5,a0000001-0000-0000-0000-000000000002,1,1620.0
6,a0000001-0000-0000-0000-000000000002,2,1620.0
7,a0000001-0000-0000-0000-000000000002,3,1620.0
8,a0000001-0000-0000-0000-000000000002,4,1620.0
9,a0000001-0000-0000-0000-000000000002,5,2700.0


In [33]:
report_merge = availability_summary.merge(
    booked_summary,
    on=["location_id", "day_of_week"],
    how="left"
)
report_merge["duration_mins"] = report_merge["duration_mins"].fillna(0)

In [36]:
report_merge

,location_id,day_of_week,available_time,duration_mins,utilization_percantage
0,a0000001-0000-0000-0000-000000000001,1,540.0,120,22.222222
1,a0000001-0000-0000-0000-000000000001,2,480.0,90,18.750000
2,a0000001-0000-0000-0000-000000000001,3,540.0,120,22.222222
3,a0000001-0000-0000-0000-000000000001,4,480.0,90,18.750000
4,a0000001-0000-0000-0000-000000000001,5,540.0,120,22.222222
5,a0000001-0000-0000-0000-000000000002,1,1620.0,60,3.703704
6,a0000001-0000-0000-0000-000000000002,2,1620.0,60,3.703704
7,a0000001-0000-0000-0000-000000000002,3,1620.0,60,3.703704
8,a0000001-0000-0000-0000-000000000002,4,1620.0,60,3.703704
9,a0000001-0000-0000-0000-000000000002,5,2700.0,60,2.222222


In [34]:
report_merge["utilization_percantage"] = (
    report_merge["duration_mins"] / report_merge["available_time"]
) * 100

In [35]:
report_merge["utilization_percantage"]

0    22.222222
1    18.750000
2    22.222222
3    18.750000
4    22.222222
5     3.703704
6     3.703704
7     3.703704
8     3.703704
9     2.222222
Name: utilization_percantage, dtype: float64

In [32]:
report_merge

,location_id,day_of_week,available_time,duration_mins,utilization_percantage
0,a0000001-0000-0000-0000-000000000001,1,540.0,120,22.222222
1,a0000001-0000-0000-0000-000000000001,2,480.0,90,18.750000
2,a0000001-0000-0000-0000-000000000001,3,540.0,120,22.222222
3,a0000001-0000-0000-0000-000000000001,4,480.0,90,18.750000
4,a0000001-0000-0000-0000-000000000001,5,540.0,120,22.222222
5,a0000001-0000-0000-0000-000000000002,1,1620.0,60,3.703704
6,a0000001-0000-0000-0000-000000000002,2,1620.0,60,3.703704
7,a0000001-0000-0000-0000-000000000002,3,1620.0,60,3.703704
8,a0000001-0000-0000-0000-000000000002,4,1620.0,60,3.703704
9,a0000001-0000-0000-0000-000000000002,5,2700.0,60,2.222222


# 3. Doctor workload leaderboard


In [22]:
pd.read_sql("SELECT * FROM appointment;", engine)

,appointment_id,patient_id,doctor_id,location_id,room_id,scheduled_at,duration_mins,appointment_type,status,telehealth_url,chief_complaint,notes,created_at
0,aa000001-0000-0000-0000-000000000001,e0000001-0000-0000-0000-000000000001,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,b0000001-0000-0000-0000-000000000001,2025-05-12 16:00:00+00:00,60,in_person,completed,None,Persistent fatigue and shortness of breath,"BP 138/88, ordered CBC and CMP",2026-04-29 02:37:01.556217+00:00
1,aa000001-0000-0000-0000-000000000002,e0000001-0000-0000-0000-000000000002,c0000001-0000-0000-0000-000000000003,a0000001-0000-0000-0000-000000000002,None,2025-05-14 17:00:00+00:00,30,telehealth,completed,https://meet.meridian.example/rm/XJ9KL2,Medication review — metformin titration,Increased metformin to 1000mg BID,2026-04-29 02:37:01.556217+00:00
2,aa000001-0000-0000-0000-000000000003,e0000001-0000-0000-0000-000000000003,c0000001-0000-0000-0000-000000000002,a0000001-0000-0000-0000-000000000001,b0000001-0000-0000-0000-000000000002,2025-05-20 21:00:00+00:00,45,in_person,completed,None,Palpitations and exertional chest tightness,ECG ordered; lipid panel requested,2026-04-29 02:37:01.556217+00:00
3,aa000001-0000-0000-0000-000000000004,e0000001-0000-0000-0000-000000000004,c0000001-0000-0000-0000-000000000004,a0000001-0000-0000-0000-000000000002,None,2025-05-21 18:00:00+00:00,30,telehealth,completed,https://meet.meridian.example/rm/AO5TR8,Type 2 diabetes quarterly check-in,HbA1c trending down; continue current regimen,2026-04-29 02:37:01.556217+00:00
4,aa000001-0000-0000-0000-000000000005,e0000001-0000-0000-0000-000000000005,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,b0000001-0000-0000-0000-000000000001,2025-05-28 16:30:00+00:00,30,in_person,completed,None,Annual wellness exam,All vitals normal; advised flu vaccine,2026-04-29 02:37:01.556217+00:00
5,aa000001-0000-0000-0000-000000000006,e0000001-0000-0000-0000-000000000006,c0000001-0000-0000-0000-000000000003,a0000001-0000-0000-0000-000000000002,b0000001-0000-0000-0000-000000000005,2025-06-02 15:30:00+00:00,60,in_person,completed,None,New patient — hypertension and osteoporosis ma...,Referred for DEXA scan; BP meds reviewed,2026-04-29 02:37:01.556217+00:00
6,aa000001-0000-0000-0000-000000000008,e0000001-0000-0000-0000-000000000008,c0000001-0000-0000-0000-000000000002,a0000001-0000-0000-0000-000000000001,b0000001-0000-0000-0000-000000000002,2026-06-12 21:30:00+00:00,45,in_person,confirmed,None,Post-cardiac stress test review,None,2026-04-29 02:37:01.556217+00:00
7,aa000001-0000-0000-0000-000000000009,e0000001-0000-0000-0000-000000000009,c0000001-0000-0000-0000-000000000004,a0000001-0000-0000-0000-000000000002,None,2025-06-05 17:00:00+00:00,30,telehealth,cancelled,None,Thyroid follow-up,Patient cancelled — rescheduling,2026-04-29 02:37:01.556217+00:00
8,aa000001-0000-0000-0000-000000000010,e0000001-0000-0000-0000-000000000010,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,b0000001-0000-0000-0000-000000000001,2025-06-04 17:00:00+00:00,30,in_person,completed,None,Follow-up — iron-deficiency anaemia,Haemoglobin improving; continue iron supplemen...,2026-04-29 02:37:01.556217+00:00
9,aa000001-0000-0000-0000-000000000007,e0000001-0000-0000-0000-000000000007,c0000001-0000-0000-0000-000000000003,a0000001-0000-0000-0000-000000000002,None,2026-06-10 16:00:00+00:00,30,telehealth,confirmed,https://meet.meridian.example/rm/DM3QW6,Anxiety follow-up,None,2026-04-29 02:37:01.556217+00:00


In [46]:
# total appointments per doctor
pd.read_sql("""
SELECT doctor_id, COUNT(*) AS total_appointments
FROM appointment
GROUP BY doctor_id
""", engine)

,doctor_id,total_appointments
0,c0000001-0000-0000-0000-000000000003,3
1,c0000001-0000-0000-0000-000000000002,2
2,c0000001-0000-0000-0000-000000000004,2
3,c0000001-0000-0000-0000-000000000001,3


In [40]:
pd.read_sql("SELECT * FROM invoice;", engine)

,invoice_id,appointment_id,patient_id,subtotal,discount,tax,total_amount,amount_paid,status,issued_at,due_date,notes,created_at
0,dd000001-0000-0000-0000-000000000001,aa000001-0000-0000-0000-000000000001,e0000001-0000-0000-0000-000000000001,390.0,0.0,0.0,390.0,390.0,paid,2025-05-13 00:00:00+00:00,2025-06-12,None,2026-04-29 02:37:01.556217+00:00
1,dd000001-0000-0000-0000-000000000002,aa000001-0000-0000-0000-000000000002,e0000001-0000-0000-0000-000000000002,120.0,0.0,0.0,120.0,120.0,paid,2025-05-15 00:00:00+00:00,2025-06-14,None,2026-04-29 02:37:01.556217+00:00
2,dd000001-0000-0000-0000-000000000003,aa000001-0000-0000-0000-000000000003,e0000001-0000-0000-0000-000000000003,310.0,0.0,0.0,310.0,248.0,partially_paid,2025-05-21 00:00:00+00:00,2025-06-20,None,2026-04-29 02:37:01.556217+00:00
3,dd000001-0000-0000-0000-000000000004,aa000001-0000-0000-0000-000000000004,e0000001-0000-0000-0000-000000000004,165.0,16.5,0.0,148.5,148.5,paid,2025-05-22 00:00:00+00:00,2025-06-21,10% courtesy discount applied,2026-04-29 02:37:01.556217+00:00
4,dd000001-0000-0000-0000-000000000005,aa000001-0000-0000-0000-000000000005,e0000001-0000-0000-0000-000000000005,200.0,0.0,15.0,215.0,100.0,partially_paid,2025-05-29 00:00:00+00:00,2025-06-28,State tax applied,2026-04-29 02:37:01.556217+00:00
5,dd000001-0000-0000-0000-000000000006,aa000001-0000-0000-0000-000000000006,e0000001-0000-0000-0000-000000000006,485.0,0.0,0.0,485.0,388.0,partially_paid,2025-06-03 00:00:00+00:00,2025-07-02,None,2026-04-29 02:37:01.556217+00:00
6,dd000001-0000-0000-0000-000000000010,aa000001-0000-0000-0000-000000000010,e0000001-0000-0000-0000-000000000010,205.0,0.0,0.0,205.0,150.0,partially_paid,2025-06-05 00:00:00+00:00,2025-07-04,None,2026-04-29 02:37:01.556217+00:00


In [56]:
pd.read_sql("""
SELECT appointment.doctor_id,
       SUM(invoice.total_amount) AS total_revenue,
       RANK() OVER (ORDER BY SUM(invoice.total_amount) DESC) AS doctor_rank
FROM appointment 
INNER JOIN invoice 
ON appointment.appointment_id = invoice.appointment_id
GROUP BY appointment.doctor_id
""", engine)

,doctor_id,total_revenue,doctor_rank
0,c0000001-0000-0000-0000-000000000001,810.0,1
1,c0000001-0000-0000-0000-000000000003,605.0,2
2,c0000001-0000-0000-0000-000000000002,310.0,3
3,c0000001-0000-0000-0000-000000000004,148.5,4


# 4.No-show rate analysis



In [54]:
pd.read_sql("""
SELECT 
    doctor_id,
    location_id,
    EXTRACT(YEAR FROM scheduled_at) AS year,
    EXTRACT(MONTH FROM scheduled_at) AS month,

    COUNT(*) AS total_appointments,

    COUNT(*) FILTER (WHERE status = 'cancelled') AS cancelled_appointments,

    (COUNT(*) FILTER (WHERE status = 'cancelled') * 100.0 / COUNT(*)) 
        AS no_show_percentage

FROM appointment
GROUP BY doctor_id, location_id, year, month
ORDER BY year, month;
""", engine)

,doctor_id,location_id,year,month,total_appointments,cancelled_appointments,no_show_percentage
0,c0000001-0000-0000-0000-000000000004,a0000001-0000-0000-0000-000000000002,2025.0,5.0,1,0,0.0
1,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,2025.0,5.0,2,0,0.0
2,c0000001-0000-0000-0000-000000000002,a0000001-0000-0000-0000-000000000001,2025.0,5.0,1,0,0.0
3,c0000001-0000-0000-0000-000000000003,a0000001-0000-0000-0000-000000000002,2025.0,5.0,1,0,0.0
4,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,2025.0,6.0,1,0,0.0
5,c0000001-0000-0000-0000-000000000004,a0000001-0000-0000-0000-000000000002,2025.0,6.0,1,1,100.0
6,c0000001-0000-0000-0000-000000000003,a0000001-0000-0000-0000-000000000002,2025.0,6.0,1,0,0.0
7,c0000001-0000-0000-0000-000000000002,a0000001-0000-0000-0000-000000000001,2026.0,6.0,1,0,0.0
8,c0000001-0000-0000-0000-000000000003,a0000001-0000-0000-0000-000000000002,2026.0,6.0,1,0,0.0


# 5.Overlapping appointment detector (should return none)

In [58]:
pd.read_sql("SELECT * FROM doctor_availability;", engine)

,availability_id,doctor_id,location_id,day_of_week,slot_start,slot_end,slot_date,is_booked,is_blocked,created_at
0,d0000001-0000-0000-0000-000000000001,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,1,09:00:00,12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00
1,d0000001-0000-0000-0000-000000000002,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,3,09:00:00,12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00
2,d0000001-0000-0000-0000-000000000003,c0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,5,09:00:00,12:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00
3,d0000001-0000-0000-0000-000000000004,c0000001-0000-0000-0000-000000000002,a0000001-0000-0000-0000-000000000001,2,13:00:00,17:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00
4,d0000001-0000-0000-0000-000000000005,c0000001-0000-0000-0000-000000000002,a0000001-0000-0000-0000-000000000001,4,13:00:00,17:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00
5,d0000001-0000-0000-0000-000000000006,c0000001-0000-0000-0000-000000000003,a0000001-0000-0000-0000-000000000002,1,08:00:00,17:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00
6,d0000001-0000-0000-0000-000000000007,c0000001-0000-0000-0000-000000000003,a0000001-0000-0000-0000-000000000002,2,08:00:00,17:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00
7,d0000001-0000-0000-0000-000000000008,c0000001-0000-0000-0000-000000000003,a0000001-0000-0000-0000-000000000002,3,08:00:00,17:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00
8,d0000001-0000-0000-0000-000000000009,c0000001-0000-0000-0000-000000000003,a0000001-0000-0000-0000-000000000002,4,08:00:00,17:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00
9,d0000001-0000-0000-0000-000000000010,c0000001-0000-0000-0000-000000000003,a0000001-0000-0000-0000-000000000002,5,08:00:00,17:00:00,None,False,False,2026-04-29 02:37:01.556217+00:00


In [59]:
pd.read_sql("SELECT * FROM room;", engine)

,room_id,location_id,room_number,room_type,capacity,is_active,created_at
0,b0000001-0000-0000-0000-000000000001,a0000001-0000-0000-0000-000000000001,101,exam,1,True,2026-04-29 02:37:01.556217+00:00
1,b0000001-0000-0000-0000-000000000002,a0000001-0000-0000-0000-000000000001,102,exam,1,True,2026-04-29 02:37:01.556217+00:00
2,b0000001-0000-0000-0000-000000000003,a0000001-0000-0000-0000-000000000001,201,lab,4,True,2026-04-29 02:37:01.556217+00:00
3,b0000001-0000-0000-0000-000000000004,a0000001-0000-0000-0000-000000000001,202,imaging,2,True,2026-04-29 02:37:01.556217+00:00
4,b0000001-0000-0000-0000-000000000005,a0000001-0000-0000-0000-000000000002,101,exam,1,True,2026-04-29 02:37:01.556217+00:00
5,b0000001-0000-0000-0000-000000000006,a0000001-0000-0000-0000-000000000002,102,exam,1,True,2026-04-29 02:37:01.556217+00:00
6,b0000001-0000-0000-0000-000000000007,a0000001-0000-0000-0000-000000000002,201,procedure,2,True,2026-04-29 02:37:01.556217+00:00


In [47]:
pd.read_sql("""
SELECT 
    availability1.availability_id AS appointment1,
    availability2.availability_id AS appointment2,
    'Overlapping Appointment' AS status
FROM doctor_availability availability1
INNER JOIN doctor_availability availability2
  ON availability1.doctor_id = availability2.doctor_id
 AND availability1.slot_date = availability2.slot_date
 AND availability1.availability_id < availability2.availability_id
 AND availability1.is_booked = TRUE
 AND availability2.is_booked = TRUE
 AND availability1.slot_start < availability2.slot_end --Appointment1 starts before appointment 2 ends
 AND availability1.slot_end > availability2.slot_start; --Appointment 2 ends after appointment 2 started
""",engine)

,appointment1,appointment2,status


# 6. Outstanding balances report

In [48]:
pd.read_sql("SELECT * FROM insurance_claim;", engine)

,claim_id,invoice_id,patient_id,payer_name,payer_id,member_id,group_id,claim_number,submitted_at,adjudicated_at,status,approved_amount,denied_amount,denial_reason,notes,created_at
0,ff000001-0000-0000-0000-000000000001,dd000001-0000-0000-0000-000000000001,e0000001-0000-0000-0000-000000000001,Blue Shield of CA,BSC-NPI-77001,BSC-100001,GRP-4400,BSC-CLM-20250514-001,2025-05-14 07:00:00+00:00,2025-05-24 07:00:00+00:00,paid,312.0,0.0,None,Primary claim — paid at 80% of allowed amount,2026-04-29 02:37:01.556217+00:00
1,ff000001-0000-0000-0000-000000000002,dd000001-0000-0000-0000-000000000002,e0000001-0000-0000-0000-000000000002,Aetna,AET-NPI-88002,AET-200002,GRP-7710,AET-CLM-20250516-001,2025-05-16 07:00:00+00:00,2025-05-20 07:00:00+00:00,paid,120.0,0.0,None,Telehealth claim — approved in full,2026-04-29 02:37:01.556217+00:00
2,ff000001-0000-0000-0000-000000000003,dd000001-0000-0000-0000-000000000003,e0000001-0000-0000-0000-000000000003,Kaiser Permanente,KP-NPI-33001,KP-300003,GRP-9920,KP-CLM-20250522-001,2025-05-22 07:00:00+00:00,2025-05-27 07:00:00+00:00,approved,248.0,62.0,ECG interpretation fee ($62) not covered under...,Appealing ECG denial,2026-04-29 02:37:01.556217+00:00
3,ff000001-0000-0000-0000-000000000004,dd000001-0000-0000-0000-000000000003,e0000001-0000-0000-0000-000000000003,Kaiser Permanente,KP-NPI-33001,KP-300003,GRP-9920,KP-CLM-20250605-002,2025-06-05 07:00:00+00:00,NaT,appealed,NaN,NaN,None,Appeal submitted with clinical necessity docum...,2026-04-29 02:37:01.556217+00:00
4,ff000001-0000-0000-0000-000000000005,dd000001-0000-0000-0000-000000000006,e0000001-0000-0000-0000-000000000006,Medicare,MCR-NPI-00001,MCR-600006,GRP-0010,MCR-CLM-20250604-001,2025-06-04 07:00:00+00:00,2025-06-09 07:00:00+00:00,paid,388.0,97.0,Patient responsibility (20% co-insurance) not ...,None,2026-04-29 02:37:01.556217+00:00


In [49]:
pd.read_sql("SELECT * FROM invoice;", engine)

,invoice_id,appointment_id,patient_id,subtotal,discount,tax,total_amount,amount_paid,status,issued_at,due_date,notes,created_at
0,dd000001-0000-0000-0000-000000000001,aa000001-0000-0000-0000-000000000001,e0000001-0000-0000-0000-000000000001,390.0,0.0,0.0,390.0,390.0,paid,2025-05-13 00:00:00+00:00,2025-06-12,None,2026-04-29 02:37:01.556217+00:00
1,dd000001-0000-0000-0000-000000000002,aa000001-0000-0000-0000-000000000002,e0000001-0000-0000-0000-000000000002,120.0,0.0,0.0,120.0,120.0,paid,2025-05-15 00:00:00+00:00,2025-06-14,None,2026-04-29 02:37:01.556217+00:00
2,dd000001-0000-0000-0000-000000000003,aa000001-0000-0000-0000-000000000003,e0000001-0000-0000-0000-000000000003,310.0,0.0,0.0,310.0,248.0,partially_paid,2025-05-21 00:00:00+00:00,2025-06-20,None,2026-04-29 02:37:01.556217+00:00
3,dd000001-0000-0000-0000-000000000004,aa000001-0000-0000-0000-000000000004,e0000001-0000-0000-0000-000000000004,165.0,16.5,0.0,148.5,148.5,paid,2025-05-22 00:00:00+00:00,2025-06-21,10% courtesy discount applied,2026-04-29 02:37:01.556217+00:00
4,dd000001-0000-0000-0000-000000000005,aa000001-0000-0000-0000-000000000005,e0000001-0000-0000-0000-000000000005,200.0,0.0,15.0,215.0,100.0,partially_paid,2025-05-29 00:00:00+00:00,2025-06-28,State tax applied,2026-04-29 02:37:01.556217+00:00
5,dd000001-0000-0000-0000-000000000006,aa000001-0000-0000-0000-000000000006,e0000001-0000-0000-0000-000000000006,485.0,0.0,0.0,485.0,388.0,partially_paid,2025-06-03 00:00:00+00:00,2025-07-02,None,2026-04-29 02:37:01.556217+00:00
6,dd000001-0000-0000-0000-000000000010,aa000001-0000-0000-0000-000000000010,e0000001-0000-0000-0000-000000000010,205.0,0.0,0.0,205.0,150.0,partially_paid,2025-06-05 00:00:00+00:00,2025-07-04,None,2026-04-29 02:37:01.556217+00:00
